# ProjectPuente Kaggle Sequential Training (CBK → CEB)

This notebook is built for **Kaggle GPU (T4 x2)** and runs **two sequential LoRA trainings**:
1. Chavacano (CBK → EN)
2. Cebuano (CEB → EN)

It assumes project root is `/kaggle/working/ProjectPuente` and launcher is `notebooks/scripts/run_kaggle_phase_a_training.sh`.

In [ ]:
import os
import sys

try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is not available. Ensure Kaggle runtime is active.") from exc

print(f"python={sys.version.split()[0]}")
print(f"cwd={os.getcwd()}")
print(f"cuda_available={torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit("ERROR: Enable Kaggle Accelerator = GPU T4 x2, then rerun this cell.")
print(f"gpu_count={torch.cuda.device_count()}")
print(f"active_gpu={torch.cuda.get_device_name(0)}")

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente

if [[ ! -d "$PRJ/.git" ]]; then
  git clone https://github.com/4ldrian01/ProjectPuente.git "$PRJ"
fi

cd "$PRJ"
git fetch --all --prune
git checkout development
git pull --ff-only origin development

echo "Project root: $PRJ"
ls -lah "$PRJ"

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente
cd "$PRJ"

python -m pip install --upgrade pip wheel setuptools
python -m pip install -r notebooks/scripts/requirements_colab.txt

python - <<'PY'
import torch, transformers, peft, datasets, accelerate, huggingface_hub
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('peft', peft.__version__)
print('datasets', datasets.__version__)
print('accelerate', accelerate.__version__)
print('huggingface_hub', huggingface_hub.__version__)
PY

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente
mkdir -p "$PRJ/.secrets"
chmod 700 "$PRJ/.secrets"

if [[ -n "${HF_TOKEN:-}" ]]; then
  printf '%s\n' "$HF_TOKEN" > "$PRJ/.secrets/hf_token"
  chmod 600 "$PRJ/.secrets/hf_token"
  echo "HF token written to $PRJ/.secrets/hf_token"
else
  echo "HF_TOKEN env var not set. Optional: set it in Python first:"
  echo "import os; os.environ['HF_TOKEN']='hf_xxx'"
fi

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente
SPLIT_ROOT="$PRJ/datasets/processed/80-10-10_split"

echo '=== CBK counts (01_chavacano, LATEST set) ==='
wc -l \
  "$SPLIT_ROOT/01_chavacano/LATEST_cbk_en_train.jsonl" \
  "$SPLIT_ROOT/01_chavacano/LATEST_cbk_en_val.jsonl" \
  "$SPLIT_ROOT/01_chavacano/LATEST_cbk_en_test.jsonl"

echo
echo '=== CEB counts (02_cebuano, LATEST set) ==='
wc -l \
  "$SPLIT_ROOT/02_cebuano/LATEST_ceb_en_train.jsonl" \
  "$SPLIT_ROOT/02_cebuano/LATEST_ceb_en_val.jsonl" \
  "$SPLIT_ROOT/02_cebuano/LATEST_ceb_en_test.jsonl"

## Run 1: Chavacano (CBK → EN)
Start training via launcher with `nohup`.

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente
cd "$PRJ"

export PUENTE_PROJECT_ROOT="$PRJ"
export PUENTE_DRIVE_ROOT="$PRJ"
export PUENTE_ARTIFACT_ROOT="$PRJ"
export PUENTE_SOURCE_FLORES=cbk_Latn
export PUENTE_TARGET_FLORES=eng_Latn
export PUENTE_SOURCE_TRANSLATION_KEY=cbk
export PUENTE_TARGET_TRANSLATION_KEY=en
export PUENTE_DATASET_REL_DIR=datasets/processed/80-10-10_split/01_chavacano
export PUENTE_TRAIN_FILENAME=LATEST_cbk_en_train.jsonl
export PUENTE_EVAL_FILENAME=LATEST_cbk_en_val.jsonl
export PUENTE_TEST_FILENAME=LATEST_cbk_en_test.jsonl
export PUENTE_RUN_NAME=lora-cbk-full-kaggle
export PUENTE_REQUIRE_GPU=true
export PUENTE_SAVE_STEPS=500
export PUENTE_EPOCHS=3
export PUENTE_BATCH_SIZE_TRAIN=4
export PUENTE_BATCH_SIZE_EVAL=4
export PUENTE_GRAD_ACCUM_STEPS=4
export PUENTE_LR=0.0002

LOG="$PRJ/outputs/$PUENTE_RUN_NAME/train_$(date +%Y%m%d_%H%M%S).log"
mkdir -p "$(dirname "$LOG")"

nohup bash notebooks/scripts/run_kaggle_phase_a_training.sh > "$LOG" 2>&1 &
PID=$!

echo "$PID" > /tmp/puente_cbk_pid
echo "$LOG" > /tmp/puente_cbk_log

echo "Started CBK training PID=$PID"
echo "CBK log=$LOG"
sleep 15
sed -n '1,220p' "$LOG"

In [ ]:
%%bash
set -euo pipefail

LOG="$(cat /tmp/puente_cbk_log)"
echo "Tailing CBK log: $LOG"
echo "Use Kernel Interrupt to stop tail when you want to continue."
tail -f "$LOG"

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente
PID="$(cat /tmp/puente_cbk_pid)"
LOG="$(cat /tmp/puente_cbk_log)"

if kill -0 "$PID" 2>/dev/null; then
  echo "CBK process still running (PID=$PID). Re-run the tail cell."
  exit 1
fi

echo "CBK process completed: PID=$PID"
tail -n 80 "$LOG"

CBK_ADAPTER_DIR="$PRJ/models/lora_adapters/lora-cbk-full-kaggle"
CBK_CKPT_DIR="$PRJ/models/checkpoints/lora-cbk-full-kaggle"

test -d "$CBK_ADAPTER_DIR"
test -f "$CBK_ADAPTER_DIR/adapter_config.json"
echo "CBK adapter ready at: $CBK_ADAPTER_DIR"
echo "CBK checkpoints at: $CBK_CKPT_DIR"
find "$CBK_CKPT_DIR" -maxdepth 1 -type d -name 'checkpoint-*' | sort | tail -n 5 || true

## Between runs: cleanup and GPU memory reset

In [ ]:
import gc

try:
    import torch
except Exception:
    torch = None

gc.collect()
if torch is not None and torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
print('CUDA cache cleared. Ready for CEB run.')

## Run 2: Cebuano (CEB → EN)
Uses `LATEST_ceb_en_*` by default, with automatic fallback to `FINAL_ceb_en_*` if LATEST files are absent.

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente
cd "$PRJ"

export PUENTE_PROJECT_ROOT="$PRJ"
export PUENTE_DRIVE_ROOT="$PRJ"
export PUENTE_ARTIFACT_ROOT="$PRJ"
export PUENTE_SOURCE_FLORES=ceb_Latn
export PUENTE_TARGET_FLORES=eng_Latn
export PUENTE_SOURCE_TRANSLATION_KEY=ceb
export PUENTE_TARGET_TRANSLATION_KEY=en
export PUENTE_DATASET_REL_DIR=datasets/processed/80-10-10_split/02_cebuano
export PUENTE_TRAIN_FILENAME=LATEST_ceb_en_train.jsonl
export PUENTE_EVAL_FILENAME=LATEST_ceb_en_val.jsonl
export PUENTE_TEST_FILENAME=LATEST_ceb_en_test.jsonl

if [[ ! -f "$PRJ/$PUENTE_DATASET_REL_DIR/$PUENTE_TRAIN_FILENAME" ]]; then
  echo 'LATEST_ceb_en_* not found. Falling back to FINAL_ceb_en_*.'
  export PUENTE_TRAIN_FILENAME=FINAL_ceb_en_train.jsonl
  export PUENTE_EVAL_FILENAME=FINAL_ceb_en_val.jsonl
  export PUENTE_TEST_FILENAME=FINAL_ceb_en_test.jsonl
fi

export PUENTE_RUN_NAME=lora-ceb-full-kaggle
export PUENTE_REQUIRE_GPU=true
export PUENTE_SAVE_STEPS=500
export PUENTE_EPOCHS=3
export PUENTE_BATCH_SIZE_TRAIN=4
export PUENTE_BATCH_SIZE_EVAL=4
export PUENTE_GRAD_ACCUM_STEPS=4
export PUENTE_LR=0.0002

LOG="$PRJ/outputs/$PUENTE_RUN_NAME/train_$(date +%Y%m%d_%H%M%S).log"
mkdir -p "$(dirname "$LOG")"

nohup bash notebooks/scripts/run_kaggle_phase_a_training.sh > "$LOG" 2>&1 &
PID=$!

echo "$PID" > /tmp/puente_ceb_pid
echo "$LOG" > /tmp/puente_ceb_log

echo "Started CEB training PID=$PID"
echo "CEB log=$LOG"
sleep 15
sed -n '1,220p' "$LOG"

In [ ]:
%%bash
set -euo pipefail

LOG="$(cat /tmp/puente_ceb_log)"
echo "Tailing CEB log: $LOG"
echo "Use Kernel Interrupt to stop tail when you want to continue."
tail -f "$LOG"

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente
PID="$(cat /tmp/puente_ceb_pid)"
LOG="$(cat /tmp/puente_ceb_log)"

if kill -0 "$PID" 2>/dev/null; then
  echo "CEB process still running (PID=$PID). Re-run the tail cell."
  exit 1
fi

echo "CEB process completed: PID=$PID"
tail -n 80 "$LOG"

CEB_ADAPTER_DIR="$PRJ/models/lora_adapters/lora-ceb-full-kaggle"
CEB_CKPT_DIR="$PRJ/models/checkpoints/lora-ceb-full-kaggle"

test -d "$CEB_ADAPTER_DIR"
test -f "$CEB_ADAPTER_DIR/adapter_config.json"
echo "CEB adapter ready at: $CEB_ADAPTER_DIR"
echo "CEB checkpoints at: $CEB_CKPT_DIR"
find "$CEB_CKPT_DIR" -maxdepth 1 -type d -name 'checkpoint-*' | sort | tail -n 5 || true

In [ ]:
%%bash
set -euo pipefail

PRJ=/kaggle/working/ProjectPuente

echo '=== Final artifact summary ==='
echo
echo '[adapters]'
du -sh "$PRJ/models/lora_adapters/lora-cbk-full-kaggle" || true
du -sh "$PRJ/models/lora_adapters/lora-ceb-full-kaggle" || true

echo
echo '[checkpoints]'
du -sh "$PRJ/models/checkpoints/lora-cbk-full-kaggle" || true
du -sh "$PRJ/models/checkpoints/lora-ceb-full-kaggle" || true

echo
echo '[metrics/config outputs]'
ls -lah "$PRJ/outputs/lora-cbk-full-kaggle" || true
ls -lah "$PRJ/outputs/lora-ceb-full-kaggle" || true